In [ ]:
import numpy as np
import pandas as pd

# Task 0
Read the dataset from csv file & perform data cleaning - remove all rows, which contains `?` in some columns.
Also check for data correctness (salary & salary $K).

In [20]:
import pandas as pd

# 1. Read the dataset
df = pd.read_csv("../data/adult.csv")

# Clean up column headers by removing any leading/trailing whitespace
df.columns = df.columns.str.strip()

# Print columns so you can see exactly what they are named
print("Actual columns in your dataset:", df.columns.tolist())

# 2. Clean data by removing rows containing '?'
df = df.replace("?", pd.NA).dropna()

# 3. Dynamic Correctness Check
# Let's dynamically find the column that looks like 'salary K' or 'salary $K'
salary_col = "salary"
salary_k_col = [c for c in df.columns if "salary" in c.lower() and c != "salary"]

if salary_k_col:
    target_k_col = salary_k_col[0]
    print(f"Comparing '{salary_col}' with found column '{target_k_col}'...")

    # Convert to numeric safely
    df[salary_col] = pd.to_numeric(df[salary_col], errors="coerce")
    df[target_k_col] = pd.to_numeric(df[target_k_col], errors="coerce")
    df = df.dropna()

    # Perform correctness check
    is_correct = (df[salary_col] / 1000 == df[target_k_col]).all()
    print(f"Data correctness check passed: {is_correct}")
else:
    print(
        "Could not find a second salary column. Please check the printed column list above."
    )


Actual columns in your dataset: ['Unnamed: 0', 'age', 'workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'hours-per-week', 'native-country', 'salary', 'salary K$']
Comparing 'salary' with found column 'salary K$'...
Data correctness check passed: True


# Task 1
Print the count of men and women in the dataset.

In [21]:
# Print the exact count of men and women in the dataset
gender_counts = df["sex"].value_counts()
print(gender_counts)


Series([], Name: count, dtype: int64)


# Task 2
Find the average age of men in dataset

In [22]:
# Filter the dataframe for rows where sex is 'Male', then calculate the mean of the age column
average_age_men = df.loc[df["sex"] == "Male", "age"].mean()

print(f"Average age of men: {average_age_men:.1f} years")


Average age of men: nan years


# Task 3
Get the percentage of people from Poland (native-country)

In [23]:
# Calculate the percentage of people from Poland
poland_percentage = (df["native-country"] == "Poland").mean() * 100

print(f"Percentage of people from Poland: {poland_percentage:.2f}%")


Percentage of people from Poland: nan%


# Task 4
Get the mean and standard deviation of the age for people who earn > 50K per year. After this, get it for those who earn <= 50K.

In [24]:
# Group by the salary column and calculate mean and standard deviation of age
age_stats = df.groupby("salary")["age"].agg(["mean", "std"])

print(age_stats)


Empty DataFrame
Columns: [mean, std]
Index: []


# Task 5
Check, if there are some people without higher education (education: Bachelors, Prof-school, Assoc-acdm, Assoc-voc, Masters, Doctorate), but with > 50K salary

In [25]:
# 1. Define a list of all labels considered higher education
higher_education = [
    "Bachelors",
    "Prof-school",
    "Assoc-acdm",
    "Assoc-voc",
    "Masters",
    "Doctorate",
]

# 2. Filter for people without higher education who earn >50K
# ~df['education'].isin(...) negates the selection (acts as "NOT IN")
no_higher_edu_high_salary = df[
    (~df["education"].isin(higher_education)) & (df["salary"] == ">50K")
]

# 3. Check if any rows match this criteria
has_matching_people = not no_higher_edu_high_salary.empty
matching_count = len(no_higher_edu_high_salary)

print(f"Are there people without higher education earning >50K? {has_matching_people}")
print(f"Total count: {matching_count} people")


Are there people without higher education earning >50K? False
Total count: 0 people


# Task 6
Get the statistics of age for each type of education. Use `groupby` and `describe` for this.

In [26]:
# Group by education type and generate full descriptive statistics for age
education_age_stats = df.groupby("education")["age"].describe()

print(education_age_stats)


Empty DataFrame
Columns: [count, mean, std, min, 25%, 50%, 75%, max]
Index: []


# Task 7
Compare the married and non-married men salaries. Who earns more? (>50K or <=50K)
Married men are those, whom `marital-status` starts with "Married". Others are not.

In [27]:
import pandas as pd

# 1. Filter for men only
men_df = df[df["sex"] == "Male"].copy()

# 2. Vectorized check to see if marital status starts with "Married"
# This creates a boolean column: True for married, False for non-married
men_df["is_married"] = men_df["marital-status"].str.startswith("Married")

# 3. Cross-tabulate 'is_married' with 'salary' to get counts, then normalize to see percentages
salary_comparison = pd.crosstab(
    men_df["is_married"], men_df["salary"], normalize="index"
) * 100

print(salary_comparison)


Empty DataFrame
Columns: []
Index: []


# Task 8
Get the max hours per week some person works. How many people works the same amount of hours per week?

In [28]:
# 1. Find the maximum value in the hours-per-week column
max_hours = df["hours-per-week"].max()

# 2. Count how many people work this maximum number of hours
# We sum a boolean mask where True equals 1 and False equals 0
max_hours_workers_count = (df["hours-per-week"] == max_hours).sum()

print(f"Maximum hours worked per week: {max_hours} hours")
print(f"Number of people working {max_hours} hours/week: {max_hours_workers_count}")


Maximum hours worked per week: nan hours
Number of people working nan hours/week: 0


# Task 9
Analyze the correlation between data in dataset. Understand connected fields in it and print highlight thier connection.

In [29]:
import pandas as pd

# 1. Select only numeric columns dynamically
numeric_df = df.select_dtypes(include=["number"])

# 2. Compute the Pearson correlation matrix natively
correlation_matrix = numeric_df.corr()

# 3. Display the raw correlation matrix
print("--- Correlation Matrix ---")
print(correlation_matrix)

# 4. Highlight the strongest connections (excluding self-correlation of 1.0)
# Unstack the matrix, remove self-correlations, and sort by absolute strength
stacked_corr = correlation_matrix.unstack()
strong_connections = (
    stacked_corr[stacked_corr.index.get_level_values(0) != stacked_corr.index.get_level_values(1)]
    .abs()
    .sort_values(ascending=False)
)

print("\n--- Strongest Connected Fields (by Absolute Value) ---")
print(strong_connections.head(6))


--- Correlation Matrix ---
                Unnamed: 0  age  hours-per-week  salary  salary K$
Unnamed: 0             NaN  NaN             NaN     NaN        NaN
age                    NaN  NaN             NaN     NaN        NaN
hours-per-week         NaN  NaN             NaN     NaN        NaN
salary                 NaN  NaN             NaN     NaN        NaN
salary K$              NaN  NaN             NaN     NaN        NaN

--- Strongest Connected Fields (by Absolute Value) ---
Unnamed: 0  age              NaN
            hours-per-week   NaN
            salary           NaN
            salary K$        NaN
age         Unnamed: 0       NaN
            hours-per-week   NaN
dtype: float64
